
## 🧠 Multimodal Learning Practice – Summary

### 📌 Task

The goal is to generate **text descriptions (captions)** from **image inputs**, i.e., image-to-text generation.

常用： BLIP/BLIP-2 可以做 image captioning、VQA、对话式视觉问答；CLIP 则更偏“图文对齐/检索”。and LLaVA  
---

### single model in， single model out 
- text to image： stable diffusion model GPT/DALLE ..

  code: images.generate
    result = client.images.generate(
    model="gpt-image-1",
    prompt="A watercolor painting of a red fox in a snowy forest") 

- image to text，image caption， VQA， document  understanding (given the doc, query the information ) 


### multi model in， single model out 

- （text + image） - text： image analysis ， image chat， grounding-based QA 
  - General the image encoder will encode to the image tokens，as LLaVA、BLIP-2

  code: response.create 

   model="gpt-4.1",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Describe this image in detail."},
                {
                    "type": "input_image",
                    "image_url": "https://example.com/cat.png"
                }
            ]
        }
      ]

- （ text + image） - image ： image editing ，inpainting， instruction-based image generation 


### multi model in ， multi-model out
- （text + image） - （text + image）： given the multi-input to output both。 like generated image with the description but with the input a image and instruction。 

- Specially, the VIT is good  for image for its benefits in tokenization


### Fusion ： how to fusion 

- Early fusion ： exp image token connectate with text token together 融合深， 不同模态之间交互。 但是token多， training is more difficult。 need to identifiy the position and the different types of model 


- Late fusion: first extract the features and then concate or scoring the features on the same embedding space.   **not generation strong** 
  - image-text retrieval 
  - zero-shot classification 


- Cross-attention fusion: text attends to image, image attends to the text. BLip model .
  


### Loss defined on the 
1. Contrastive loss， 用于 CLIP 类图文对齐。 similarity of the image embeeding and the text embedding.

2. Language modeling loss， caption / VQA 生成。 autoregressive negative log-likelihood, with the cross entropy  

3。Matching loss 判断 image-text 是否匹配。 二分类。 

### How to add the positions information in the multi-model fusion 

1.  modality and type emebedding: 
tell the model the image token and the text token. 
2.  special tokens: image, patch, bos, question 
[<image>,v1​,…,vm​,<question>,t1​,…,tn​]  




### Blip models 
- Blip 
  1. Vision encoder 
  2. Text encoder 
  3. Text decoder 
  4. Image-Text Contrastive / Matching / LM 多种训练目标 


- Blip-2 
  它把冻结的视觉编码器和冻结的大语言模型连接起来，中间加一个轻量桥接模块（Q-Former），从而减少全模型训练成本。BLIP-2 可用于 captioning、VQA 和图像对话 



### The multimodel structure 

- Dual Encoder : image encoder and text encoder 对齐到同一 embedding space . CLIP


- Encoder - Decoder : such as BLIP 


- Visual Encoder + LLM  (LLaVA, BLIP-2):  BLIP-2 论文明确说它用一个轻量的 Querying Transformer（Q-Former）来 bridge frozen image encoder 和 frozen LLM . 

  - Vit 
  - 1. projector 映射到 LLM embedding space : h_{lm} =MLP(h_vision)
  - 2. Q-Former. with the learnable tokens, and then query the image token to get the language-friendly vision tokens . 重点摘要
  - VIsion token as the prefix <image> .... for the LLM
  - LLM output the text 

  - code for Q-former: 

  class SimpleQFormer(nn.Module):
      def __init__(self, num_queries=32, dim=768):
          super().__init__()
          self.query_tokens = nn.Parameter(torch.randn(1, num_queries, dim))
          self.cross_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=8, batch_first=True)

      def forward(self, vision_tokens):
          # vision_tokens: [B, N_img, D]
          B = vision_tokens.size(0)

          # 复制 query tokens 到 batch
          queries = self.query_tokens.expand(B, -1, -1)  # [B, num_queries, D]

          # query attends to image tokens
          out, _ = self.cross_attn(
              query=queries,
              key=vision_tokens,
              value=vision_tokens
          )
          return out  # [B, num_queries, D]


- Unified sequence model : for example, the early fusion 


### Data fromate 

caption {'text":..., "image": ...}
VQA { "image": ... "question" : ...., "answer": }

Robot , instruction{"image": ..."message":{"role", "content"} }


### What is the projector and Q-former that specially in the multimodel training 



### dataset formatting and multimodal instruction tuning 




### Instruction tuning for multimodal 
类似文本 LLM 的 instruction tuning，只不过数据变成：

用户问题 + 图像
assistant 回答

LLaVA 就是很典型的 visual instruction tuning。 
### 🔹 Approach 1: Pretrained Multimodal Models (e.g., BLIP)

* Use a pretrained model specifically designed for **vision-language tasks**

* Core components:

  * **Image Encoder**
    Extracts visual features from the image and converts them into embeddings
  * **Text Encoder (optional)**
    Used when additional text input is provided (e.g., VQA tasks)
  * **Text Decoder**
    Generates the output caption

* Key mechanism:

  * The decoder is initialized with a **BOS (beginning-of-sequence) token**
  * Uses **cross-attention** to interact with:

    * image embeddings (from the image encoder)
    * text embeddings (if text input is present)
  * Generates text **autoregressively** (token by token)
  

---

### 🔹 Approach 2: Encoder–Decoder Architecture (Custom / Modular)

* Build a modular pipeline:

  * **Encoder (Vision Model)**
    Encodes the image into feature representations
  * **Decoder (Language Model)**
    Generates text based on encoded features
  
  * **Fine tunning the model** 
   - with QLoRA
   - LoRA 
   - Prompt tunning  


* Compared to pretrained models:

  * More flexible and customizable
  * Requires more design and training effort
  * The performance may not be as good as the Pretrained model 

---



In [ ]:
## BLIP 
from PIL import Image 
import requests 

from transformers import  BlipProcessor, BlipForConditionalGeneration 

processor = "Salesforce/blip-image-captioning-base" 
model_name = "Salesforce/blip-image-captioning-base"
procesor  = BlipProcessor.from_pretrained(model_processor)
model = BlipForConditionalGeneration.from_pretrained(model_name) 



# 2. load image
url = "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")  

# preprocessor 
inputs = processor(images=image, return_tensors="pt")
 
# output 
output_ids = model.generate(**inputs, max_new_tokens= 30)

caption = model.decode( output_ids[0], skip_special_tokens = True )
 


In [ ]:
from PIL import Image
import requests
from transformers import BlipProcessor, BlipForQuestionAnswering

processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")

url = "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

question = "How many dogs are in the image?"

# input both the images and the text. 
inputs = processor(images=image, text=question, return_tensors="pt")
output_ids = model.generate(**inputs, max_new_tokens=10)

answer = processor.decode(output_ids[0], skip_special_tokens=True)
print("Answer:", answer) 

## Approach 1 


## Approach 2 